In [1]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/Adaptive-Token-Dictionary_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/Adaptive-Token-Dictionary_M2L8


/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
# import rasterio
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-02-07 14:56:40.032620: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only GPU 1

In [5]:
from basicsr.utils.options import yaml_load
opt = yaml_load('options/train/103_ATD_light_SRx4_finetune.yml')

In [6]:
opt['network_g']['upscale']=16

In [7]:
from basicsr.archs import build_network
model = build_network(opt['network_g']).to('cuda')


/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
# X = torch.randn(1, 7, 64, 64).to('cuda')

# outputs = model(X)
# print(outputs.shape)

In [8]:
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=opt['train']['optim_g']['lr'], weight_decay=opt['train']['optim_g']['weight_decay'], betas=opt['train']['optim_g']['betas'])
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=opt['train']['scheduler']['milestones'], gamma=opt['train']['scheduler']['gamma'])

In [9]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

# filenames = ['/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_0.tfrecords',
#              '/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_1.tfrecords',
#              '/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_2.tfrecords',
#              '/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_3.tfrecords',
#              '/content/drive/MyDrive/GeoSR_new/M2L8/SR_dataset/M2L8_SR_part_4.tfrecords']

# ds = input_pipeline(filenames, batch_size=5, is_shuffle=True, is_train=True, is_repeat=True)


# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))*0.0001
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))*0.0000275-0.2

#         axes[0].imshow(lres_img[:, :, [0, 3, 2]]*2)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, 3:0:-1]*2)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break


In [ ]:
def save_network(save_path, net, param_key='params'):
        """Save networks.

        Args:
            net (nn.Module | list[nn.Module]): Network(s) to be saved.
            param_key (str | list[str]): The parameter key(s) to save network.
                Default: 'params'.
        """
        net = net if isinstance(net, list) else [net]
        param_key = param_key if isinstance(param_key, list) else [param_key]
        assert len(net) == len(param_key), 'The lengths of net and param_key should be the same.'

        save_dict = {}
        for net_, param_key_ in zip(net, param_key):
            state_dict = net_.state_dict()
            for key, param in state_dict.items():
                if key.startswith('module.'):  # remove unnecessary 'module.'
                    key = key[7:]
                state_dict[key] = param.cpu()
            save_dict[param_key_] = state_dict

        torch.save(save_dict, save_path)

# save_network('/content/drive/MyDrive/GeoSR/ATD_x2_epoch_weights.pth', model)

In [14]:
from copy import deepcopy

def _print_different_keys_loading(crt_net, load_net, strict=True):
        """Print keys with different name or different size when loading models.

        1. Print keys with different names.
        2. If strict=False, print the same key but with different tensor size.
            It also ignore these keys with different sizes (not load).

        Args:
            crt_net (torch model): Current network.
            load_net (dict): Loaded network.
            strict (bool): Whether strictly loaded. Default: True.
        """
        crt_net = crt_net.state_dict()
        crt_net_keys = set(crt_net.keys())
        load_net_keys = set(load_net.keys())

        if crt_net_keys != load_net_keys:
            print('Current net - loaded net:')
            for v in sorted(list(crt_net_keys - load_net_keys)):
                print(f'  {v}')
            print('Loaded net - current net:')
            for v in sorted(list(load_net_keys - crt_net_keys)):
                print(f'  {v}')

        # check the size for the same keys
        if not strict:
            common_keys = crt_net_keys & load_net_keys
            for k in common_keys:
                if crt_net[k].size() != load_net[k].size():
                    print(f'Size different, ignore [{k}]: crt_net: '
                                   f'{crt_net[k].shape}; load_net: {load_net[k].shape}')
                    load_net[k + '.ignore'] = load_net.pop(k)

def load_network(net, load_path, strict=True, param_key='params'):
        """Load network.

        Args:
            load_path (str): The path of networks to be loaded.
            net (nn.Module): Network.
            strict (bool): Whether strictly loaded.
            param_key (str): The parameter key of loaded network. If set to
                None, use the root 'path'.
                Default: 'params'.
        """
        load_net = torch.load(load_path, map_location=lambda storage, loc: storage)
        if param_key is not None:
            if param_key not in load_net and 'params' in load_net:
                param_key = 'params'
            load_net = load_net[param_key]
        # remove unnecessary 'module.'
        for k, v in deepcopy(load_net).items():
            if k.startswith('module.'):
                load_net[k[7:]] = v
                load_net.pop(k)
        _print_different_keys_loading(net, load_net, strict)
        net.load_state_dict(load_net, strict=strict)

# load_network(model, '/content/drive/MyDrive/GeoSR_new/M2L8/SR_pretrained_weights/ATD_x4_epoch_weights_M2L8.pth', strict=False, param_key='params')
load_network(model, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/ATD_x4_epoch_weights_M2L8_new.pth', strict=False, param_key='params')
# load_network(model, '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_epoch_weights_M2L8_new.pth', strict=False, param_key='params')

Size different, ignore [upsample.0.weight]: crt_net: torch.Size([1792, 48, 3, 3]); load_net: torch.Size([112, 48, 3, 3])
Size different, ignore [upsample.0.bias]: crt_net: torch.Size([1792]); load_net: torch.Size([112])


# SR Training

In [ ]:
total_epochs = 30

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']

ds = input_pipeline(filenames, batch_size=8, is_shuffle=True, is_train=True, is_repeat=False)

for epoch in range(total_epochs):
    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
        
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

        optimizer.zero_grad()
        output = model(lr)

        l_total = 0

        # pixel loss
        l_pix = torch.nn.L1Loss()(output, hr)
        l_total += l_pix

        l_total.backward()
        optimizer.step()

        if step % 500 == 0:
            print(f'Step {step}, Loss: {l_total.item()}')

    print(f'Epoch {epoch}')
    save_network('/glade/derecho/scratch/lizhili/m2l8/ATD_x16_epoch_weights_M2L8_brighter.pth', model)

2026-02-07 14:59:59.422349: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Step 0, Loss: 0.46643176674842834
Step 500, Loss: 0.06824063509702682
Step 1000, Loss: 0.07397635281085968


2026-02-07 15:11:28.299543: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 0
Step 0, Loss: 0.05598779395222664
Step 500, Loss: 0.06322716176509857
Step 1000, Loss: 0.07143130153417587


2026-02-07 15:22:21.351083: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

Epoch 3
Step 0, Loss: 0.048705026507377625
Step 500, Loss: 0.05716985464096069
Step 1000, Loss: 0.07418670505285263
Epoch 4
Step 0, Loss: 0.045735497027635574
Step 500, Loss: 0.05422614887356758
Step 1000, Loss: 0.06950882077217102
Epoch 5
Step 0, Loss: 0.05135216936469078
Step 500, Loss: 0.06035754457116127
Step 1000, Loss: 0.07738380879163742
Epoch 6
Step 0, Loss: 0.041341621428728104
Step 500, Loss: 0.05125860869884491
Step 1000, Loss: 0.060055240988731384


2026-02-07 16:27:16.010192: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 7
Step 0, Loss: 0.050985440611839294
Step 500, Loss: 0.05913775414228439
Step 1000, Loss: 0.07491008192300797
Epoch 8
Step 0, Loss: 0.05263005197048187
Step 500, Loss: 0.047242678701877594
Step 1000, Loss: 0.060996949672698975
Epoch 9
Step 0, Loss: 0.05185432359576225
Step 500, Loss: 0.04491636902093887
Step 1000, Loss: 0.07302484661340714
Epoch 10
Step 0, Loss: 0.05275674909353256
Step 500, Loss: 0.05011000111699104
Step 1000, Loss: 0.07384683936834335
Epoch 11
Step 0, Loss: 0.04363906756043434
Step 500, Loss: 0.06017839163541794
Step 1000, Loss: 0.07649292796850204
Epoch 12
Step 0, Loss: 0.05878528952598572
Step 500, Loss: 0.04691622406244278
Step 1000, Loss: 0.06346041709184647
Epoch 13
Step 0, Loss: 0.06313919275999069
Step 500, Loss: 0.050788722932338715
Step 1000, Loss: 0.07768630236387253
Epoch 14
Step 0, Loss: 0.05322076752781868
Step 500, Loss: 0.04885919764637947
Step 1000, Loss: 0.06835732609033585


2026-02-07 17:53:45.797725: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 15
Step 0, Loss: 0.0459289588034153
Step 500, Loss: 0.05034453794360161
Step 1000, Loss: 0.07130095362663269
Epoch 16
Step 0, Loss: 0.04378805309534073
Step 500, Loss: 0.05540313199162483
Step 1000, Loss: 0.07442039251327515
Epoch 17
Step 0, Loss: 0.04129250720143318
Step 500, Loss: 0.06550556421279907
Step 1000, Loss: 0.07249605655670166
Epoch 18
Step 0, Loss: 0.04820908233523369
Step 500, Loss: 0.04760392755270004
Step 1000, Loss: 0.07679447531700134
Epoch 19
Step 0, Loss: 0.0526779405772686
Step 500, Loss: 0.04985005781054497
Step 1000, Loss: 0.06360311061143875
Epoch 20
Step 0, Loss: 0.05026513710618019
Step 500, Loss: 0.05429326742887497
Step 1000, Loss: 0.06531678140163422
Epoch 21
Step 0, Loss: 0.047588590532541275
Step 500, Loss: 0.04490235075354576
Step 1000, Loss: 0.06387799978256226
Epoch 22
Step 0, Loss: 0.05961564555764198
Step 500, Loss: 0.05255027487874031
Step 1000, Loss: 0.06556367129087448
Epoch 23
Step 0, Loss: 0.05500193312764168
Step 500, Loss: 0.049569390714

# Downstream SR Finetuning

In [15]:
def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):
    num_test = num_sample-num_training

    optimizer = optim.AdamW(model.parameters(), lr=opt['train']['optim_g']['lr'], weight_decay=opt['train']['optim_g']['weight_decay'], betas=opt['train']['optim_g']['betas'])
    load_network(model, '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_epoch_weights_M2L8_new.pth', strict=False, param_key='params')

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch

    #--------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
            lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

            optimizer.zero_grad()
            output = model(lr)

            l_total = 0

            # pixel loss
            l_pix = torch.nn.L1Loss()(output, hr)
            l_total += l_pix

            l_total.backward()
            optimizer.step()

            if step % 50 == 0:
                print(f'Step {step}, Loss: {l_total.item()}')

        # torch.save(model.state_dict(), finetuned_model)
        save_network(finetuned_model, model)
    #--------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, _, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        with torch.no_grad():
            output = model(lr)
        output = output.detach().cpu().numpy()
        # output = np.clip(output, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        output[output<0] = 0
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(lr_show[:, :, [0,3,2]])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(output_show[:, :, 3:0:-1]*0.0000275-0.2)
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()


    # Close writer
    writer.close()


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_ATD_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_ATD_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)



In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_ATD_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_ATD_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)



In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/ATD_x16_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_ATD_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)